In [3]:

import json, urllib.request, urllib.parse, urllib.error, time, random, re, sys, os

# Wczytaj produkty
with open('/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/no_img_fix_needed.json') as f:
    all_products = json.load(f)

# Pomiń aceton
products = [p for p in all_products if p['_id'] != 'product-p0196018']
print(f"Liczba produktów do przetworzenia: {len(products)}")

# Sprawdź dostępność requests
try:
    import requests
    print("requests: OK")
except ImportError:
    print("requests: brak — użyję urllib")

# Test połączenia z Sanity
SANITY_TOKEN = "sk_REMOVED_FOR_SECURITY"
SANITY_PROJECT = "nzcwegq7"
SANITY_DATASET = "production"

test_url = f"https://{SANITY_PROJECT}.api.sanity.io/v2023-08-01/data/query/{SANITY_DATASET}?query=*[_type==\"product\"][0]._id"
req = urllib.request.Request(test_url, headers={"Authorization": f"Bearer {SANITY_TOKEN}"})
try:
    resp = urllib.request.urlopen(req, timeout=10).read()
    data = json.loads(resp)
    print(f"Sanity połączenie OK: {data}")
except Exception as e:
    print(f"Sanity błąd: {e}")


Liczba produktów do przetworzenia: 126
requests: OK


Sanity połączenie OK: {'query': '*[_type=="product"][0]._id', 'result': 'prod-grunt-ceresit-ct16', 'syncTags': ['s1:8g6dOQ'], 'ms': 2}


In [7]:

import json, urllib.request, urllib.parse, urllib.error, time, random, re, os
import requests
from pathlib import Path

SANITY_TOKEN = "sk_REMOVED_FOR_SECURITY"
SANITY_PROJECT = "nzcwegq7"
SANITY_DATASET = "production"
LOG_PATH = '/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/img_fix_log.json'

SANITY_HEADERS = {
    "Authorization": f"Bearer {SANITY_TOKEN}",
    "Content-Type": "application/json"
}


def search_image_ddg(query: str, min_size=200) -> str | None:
    """Szuka zdjęcia przez DuckDuckGo Images (nieoficjalne API)."""
    try:
        # Krok 1: pobierz token vqd
        headers = {
            "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        }
        resp = requests.get("https://duckduckgo.com/", params={"q": query}, headers=headers, timeout=10)
        vqd_match = re.search(r'vqd=(["\'])([^"\']+)\1', resp.text)
        if not vqd_match:
            vqd_match = re.search(r'vqd=([\d-]+)', resp.text)
            if not vqd_match:
                return None
            vqd = vqd_match.group(1)
        else:
            vqd = vqd_match.group(2)

        # Krok 2: zapytanie o obrazy
        img_resp = requests.get(
            "https://duckduckgo.com/i.js",
            params={"l": "pl-pl", "o": "json", "q": query, "vqd": vqd, "f": ",,,,,", "p": "1"},
            headers={**headers, "Referer": "https://duckduckgo.com/"},
            timeout=10
        )
        data = img_resp.json()
        results = data.get("results", [])
        for r in results[:5]:
            w = r.get("width", 0)
            h = r.get("height", 0)
            url = r.get("image", "")
            if w >= min_size and h >= min_size and url.startswith("http"):
                return url
        return None
    except Exception as e:
        return None


def search_image_bing(query: str) -> str | None:
    """Fallback: szukaj przez Bing Images scraping."""
    try:
        encoded = urllib.parse.quote(query)
        url = f"https://www.bing.com/images/search?q={encoded}&form=HDRSC2&first=1"
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
            "Accept-Language": "pl-PL,pl;q=0.9,en;q=0.8",
        }
        resp = requests.get(url, headers=headers, timeout=12)
        # Wyciągnij pierwsze mediaUrl z JSON-like struktur
        matches = re.findall(r'"murl"\s*:\s*"(https?://[^"]+)"', resp.text)
        for m in matches[:5]:
            # Sprawdź czy URL wygląda sensownie (jpg/png/webp)
            if re.search(r'\.(jpg|jpeg|png|webp)(\?|$)', m, re.IGNORECASE):
                return m
        # Jeśli brak rozszerzenia, weź pierwszy wynik
        if matches:
            return matches[0]
        return None
    except Exception as e:
        return None


def download_image(url: str, timeout=15) -> bytes | None:
    """Pobiera obraz jako bytes."""
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
            "Accept": "image/webp,image/apng,image/*,*/*;q=0.8",
            "Referer": "https://www.google.com/"
        }
        resp = requests.get(url, headers=headers, timeout=timeout, stream=False)
        if resp.status_code == 200 and len(resp.content) > 5000:
            # Sprawdź czy to faktycznie obraz
            ct = resp.headers.get("Content-Type", "")
            if "image" in ct or len(resp.content) > 10000:
                return resp.content
        return None
    except Exception:
        return None


def upload_to_sanity(img_data: bytes, content_type: str = "image/jpeg") -> str | None:
    """Wgrywa obraz do Sanity i zwraca asset_id."""
    try:
        upload_url = f"https://{SANITY_PROJECT}.api.sanity.io/v2023-08-01/assets/images/{SANITY_DATASET}"
        resp = requests.post(
            upload_url,
            data=img_data,
            headers={
                "Authorization": f"Bearer {SANITY_TOKEN}",
                "Content-Type": content_type
            },
            timeout=30
        )
        if resp.status_code in (200, 201):
            doc = resp.json().get("document", {})
            return doc.get("_id")
        else:
            print(f"  Upload error {resp.status_code}: {resp.text[:200]}")
            return None
    except Exception as e:
        print(f"  Upload exception: {e}")
        return None


def patch_product_image(product_id: str, asset_id: str) -> bool:
    """Ustawia pole images w produkcie Sanity."""
    try:
        mutation = {
            "mutations": [{
                "patch": {
                    "id": product_id,
                    "set": {
                        "images": [{
                            "_type": "image",
                            "_key": "main",
                            "asset": {
                                "_type": "reference",
                                "_ref": asset_id
                            }
                        }]
                    }
                }
            }]
        }
        resp = requests.post(
            f"https://{SANITY_PROJECT}.api.sanity.io/v2023-08-01/data/mutate/{SANITY_DATASET}",
            json=mutation,
            headers=SANITY_HEADERS,
            timeout=15
        )
        return resp.status_code in (200, 201)
    except Exception as e:
        print(f"  Patch exception: {e}")
        return False


def load_log() -> list:
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH) as f:
            return json.load(f)
    return []


def save_log(log: list):
    with open(LOG_PATH, 'w', encoding='utf-8') as f:
        json.dump(log, f, ensure_ascii=False, indent=2)


print("Funkcje zdefiniowane OK")
print(f"Produkty do przetworzenia: {len(products)}")

# Test funkcji wyszukiwania
test = search_image_ddg("Fischer kołek rozporowy UX 8x60")
print(f"Test DDG: {test[:80] if test else 'brak'}")


Funkcje zdefiniowane OK
Produkty do przetworzenia: 126
Test DDG: brak


In [11]:

# Sprawdzam dostępność duckduckgo_search
try:
    from duckduckgo_search import DDGS
    print("duckduckgo_search: OK")
    with DDGS() as ddgs:
        results = list(ddgs.images("Fischer kołek rozporowy UX 8x60", max_results=3))
    print(f"Wyniki: {results[0] if results else 'brak'}")
except ImportError:
    print("duckduckgo_search: niedostępny")
except Exception as e:
    print(f"duckduckgo_search błąd: {e}")

# Test Bing
test_bing = search_image_bing("Fischer kołek rozporowy UX 8x60")
print(f"Bing fallback: {test_bing[:80] if test_bing else 'brak'}")


duckduckgo_search: niedostępny


Bing fallback: brak


In [15]:

import subprocess
result = subprocess.run([sys.executable, "-m", "pip", "install", "duckduckgo-search", "-q"], 
                      capture_output=True, text=True, timeout=60)
print("PIP:", result.stdout[-200:] if result.stdout else "", result.stderr[-200:] if result.stderr else "")

from duckduckgo_search import DDGS
print("Import OK")

# Test
with DDGS() as ddgs:
    results = list(ddgs.images("Fischer kołek rozporowy UX 8x60mm", max_results=3))
print(f"Wyniki ({len(results)}):")
for r in results:
    print(f"  {r.get('width')}x{r.get('height')} — {r.get('image', '')[:80]}")


PIP:  
Import OK


/tmp/ipykernel_1776/1909563233.py:10: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


RatelimitException: https://duckduckgo.com/i.js?o=json&q=Fischer+ko%C5%82ek+rozporowy+UX+8x60mm&l=us-en&vqd=4-70850390215676382755874494111208006478&p=1&f=%2C%2C%2C%2C%2C 403 Ratelimit

In [19]:

import subprocess, time

# Instalacja nowej paczki ddgs
r = subprocess.run([sys.executable, "-m", "pip", "install", "ddgs", "-q"], 
                   capture_output=True, text=True, timeout=60)
print("ddgs install:", r.returncode)

try:
    from ddgs import DDGS
    print("ddgs (nowa): OK")
    time.sleep(2)
    with DDGS() as ddgs:
        results = list(ddgs.images("Fischer kołek rozporowy UX 8x60mm", max_results=3))
    print(f"Wyniki ddgs: {len(results)}")
    for r in results[:2]:
        print(f"  {r.get('width')}x{r.get('height')} — {r.get('image','')[:80]}")
except Exception as e:
    print(f"ddgs błąd: {e}")

# Alternatywa: Unsplash-style (dla materiałów budowlanych nie zadziała dobrze)
# Sprawdź czy jest Serper
try:
    import httpx
    print("httpx: OK")
except:
    print("httpx: brak")


ddgs install: 0
ddgs błąd: No module named 'ddgs'
httpx: OK


In [23]:

import json

with open('/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/no_img_fix_needed.json') as f:
    all_products = json.load(f)

products = [p for p in all_products if p['_id'] != 'product-p0196018']
print(f"Łączna liczba: {len(products)}")

# Podział na 5 partii
batch_size = 26
batches = [products[i:i+batch_size] for i in range(0, len(products), batch_size)]
for i, b in enumerate(batches):
    print(f"Partia {i+1}: {len(b)} produktów ({b[0]['_id']} ... {b[-1]['_id']})")
    # Zapisz partię do pliku
    with open(f'/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/batch_{i+1}.json', 'w') as f:
        json.dump(b, f, ensure_ascii=False, indent=2)

print("\nPartie zapisane.")


Łączna liczba: 126
Partia 1: 26 produktów (product-p028 ... product-p057)
Partia 2: 26 produktów (product-p058 ... product-p088)
Partia 3: 26 produktów (product-p089 ... product-p118)
Partia 4: 26 produktów (product-p119 ... product-p147)
Partia 5: 22 produktów (product-p148 ... product-p176)

Partie zapisane.


In [27]:

import os, requests, json, time

# Sprawdź klucze API
env_keys = {k: v[:20]+"..." for k, v in os.environ.items() 
            if any(x in k.upper() for x in ['API', 'KEY', 'SEARCH', 'BING', 'SERPER', 'GOOGLE'])}
print("Dostępne klucze API:")
for k, v in env_keys.items():
    print(f"  {k}: {v}")

# Sprawdź /etc/environment i inne źródła
try:
    with open('/etc/environment') as f:
        print("\n/etc/environment:", f.read()[:500])
except: pass

# Sprawdź aplikacje dostępne
import subprocess
for tool in ['curl', 'wget', 'lynx']:
    r = subprocess.run(['which', tool], capture_output=True, text=True)
    print(f"{tool}: {r.stdout.strip() or 'brak'}")


Dostępne klucze API:

/etc/environment: AGENT_BROWSER_EXECUTABLE_PATH=/ms-playwright/chromium-1194/chrome-linux/chrome

curl: /usr/bin/curl
wget: /usr/bin/wget
lynx: brak


In [31]:

import requests, re, urllib.parse, time

UA = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"

def search_bing_images(query, n=3):
    """Bing Image Search przez scraping."""
    try:
        url = "https://www.bing.com/images/search"
        params = {"q": query, "form": "HDRSC2", "first": "1", "tsc": "ImageHoverTitle"}
        headers = {
            "User-Agent": UA,
            "Accept": "text/html,application/xhtml+xml",
            "Accept-Language": "pl-PL,pl;q=0.9",
            "Referer": "https://www.bing.com/",
        }
        r = requests.get(url, params=params, headers=headers, timeout=12)
        # Wyciągnij URLs z JSON-like bloków
        matches = re.findall(r'"murl"\s*:\s*"(https?://[^"]+)"', r.text)
        results = []
        for m in matches[:n*2]:
            if any(ext in m.lower() for ext in ['.jpg','.jpeg','.png','.webp']):
                results.append(m)
            elif len(results) == 0:
                results.append(m)
            if len(results) >= n:
                break
        return results
    except Exception as e:
        return []

def search_yahoo_images(query, n=3):
    """Yahoo Image Search przez scraping."""
    try:
        encoded = urllib.parse.quote(query)
        url = f"https://images.search.yahoo.com/search/images;_ylt=AwrNZo?p={encoded}&ei=UTF-8&fr=yfp-t"
        headers = {
            "User-Agent": UA,
            "Accept": "text/html",
            "Accept-Language": "pl-PL,pl;q=0.9",
        }
        r = requests.get(url, headers=headers, timeout=12)
        matches = re.findall(r'"url"\s*:\s*"(https?://[^"]+\.(jpg|jpeg|png|webp))"', r.text, re.IGNORECASE)
        return [m[0] for m in matches[:n]]
    except Exception as e:
        return []

def search_ecosia_images(query, n=3):
    """Ecosia scraping."""
    try:
        encoded = urllib.parse.quote(query)
        url = f"https://www.ecosia.org/images?q={encoded}"
        headers = {"User-Agent": UA, "Accept-Language": "pl-PL"}
        r = requests.get(url, headers=headers, timeout=12)
        # Szukaj URL-i obrazów
        matches = re.findall(r'"(https?://[^"]+\.(?:jpg|jpeg|png|webp))"', r.text, re.IGNORECASE)
        return list(set(matches[:n]))
    except Exception as e:
        return []

# Test wszystkich metod
q = "Isover Akustik wełna szklana 50mm"
print(f"Query: {q}")

r1 = search_bing_images(q)
print(f"Bing ({len(r1)}): {r1[0][:80] if r1 else 'brak'}")

time.sleep(1)
r2 = search_yahoo_images(q)
print(f"Yahoo ({len(r2)}): {r2[0][:80] if r2 else 'brak'}")

time.sleep(1)
r3 = search_ecosia_images(q)
print(f"Ecosia ({len(r3)}): {r3[0][:80] if r3 else 'brak'}")


Query: Isover Akustik wełna szklana 50mm


Bing (0): brak


Yahoo (0): brak


Ecosia (0): brak


In [35]:

import requests, re, time

session = requests.Session()
UA = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"

session.headers.update({
    "User-Agent": UA,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "pl-PL,pl;q=0.9,en;q=0.8",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
})

# Najpierw odwiedź stronę główną Bing żeby dostać cookies
try:
    r0 = session.get("https://www.bing.com/", timeout=10)
    print(f"Bing home: {r0.status_code}, cookies: {len(session.cookies)}")
    
    time.sleep(1)
    
    # Teraz szukaj obrazów
    r1 = session.get(
        "https://www.bing.com/images/search",
        params={"q": "Isover Akustik wełna szklana 50mm płyta", "form": "HDRSC2"},
        timeout=12
    )
    print(f"Bing search: {r1.status_code}, len: {len(r1.text)}")
    
    # Szukaj URL-i obrazów
    patterns = [
        r'"murl"\s*:\s*"(https?://[^"]+)"',
        r'"imgurl"\s*:\s*"(https?://[^"]+)"',
        r'iurl":"(https?://[^"]+)"',
    ]
    for p in patterns:
        m = re.findall(p, r1.text)
        print(f"Pattern {p[:20]}: {len(m)} wyników")
        if m:
            print(f"  Przykład: {m[0][:100]}")
            break
            
    # Spróbuj też Google
    time.sleep(1)
    rg = session.get(
        "https://www.google.com/search",
        params={"q": "Isover Akustik wełna szklana 50mm", "tbm": "isch", "hl": "pl"},
        headers={"Referer": "https://www.google.com/"},
        timeout=12
    )
    print(f"\nGoogle Images: {rg.status_code}, len: {len(rg.text)}")
    # Szukaj URL-i
    gm = re.findall(r'"(https?://[^"]+\.(?:jpg|jpeg|png|webp)(?:\?[^"]*)?)"', rg.text, re.IGNORECASE)
    gm2 = re.findall(r'https?://(?:encrypted-tbn\d+\.gstatic\.com/images\?[^"&\s]+)', rg.text)
    print(f"Google bezpośrednie: {len(gm)} URL-i")
    print(f"Google thumbnails: {len(gm2)}")
    if gm:
        print(f"  Przykład: {gm[0][:100]}")
    
except Exception as e:
    print(f"Błąd: {e}")


Bing home: 200, cookies: 10


Bing search: 200, len: 58980
Pattern "murl"\s*:\s*"(https: 0 wyników
Pattern "imgurl"\s*:\s*"(htt: 0 wyników
Pattern iurl":"(https?://[^": 0 wyników



Google Images: 200, len: 42524
Google bezpośrednie: 0 URL-i
Google thumbnails: 0


In [39]:

import json

# Zebrane URL-e — checkpoint po 20 produktach
url_map = {
    "product-p028": "https://www.abud.pl/hpeciai/bf9ec332eacf363d213028101d192d51/pol_pl_Kolki-rozporowe-Fischer-UX-8x50-RK-10-sztuk-17791_1.webp",
    "product-p029": "https://folnet.pl/hpeciai/3270ded56daf8ca9ab4749f9829b5a5e/pol_pl_Membrana-Dachowa-Delta-Vent-N-PLUS-18896_2.jpg",
    "product-p030": "https://katepal.fi/wp-content/uploads/2025/02/Katepal-Jazzy-ruskea.jpg",
    "product-p031": "https://sklep.sewera.pl/photo/254714/kostka-brukowa-holland-szary-6cm-bruk-bet.webp",
    "product-p032": "https://media.knauf.com/a/Wjs6iHy2hdkdAbkqyzhhZC",
    "product-p034": "https://mbdombud.pl/wp-content/uploads/2021/04/MPI_25_-30kg.jpg",
    "product-p035": "https://bednarek.sklep.pl/13935-large_default/isover-aku-plyta-5cm.jpg",
    "product-p036": "https://sklep.papiarz.pl/environment/cache/images/productGfx_638_500_500/rynna_A4.jpg",
    "product-p037": "https://suez.b-cdn.net/media/93/af/cb/1654163695/bc0fc377a1b9bed81bd23a2483f491a1.jpg",
    "product-p038": "https://klinkiernia.com.pl/6241-home_default/cegla-klinkierowa-lode-pelna-kl50.jpg",
    "product-p039": "https://productimages.etrusted.com/products/prt-2d4b4f1e-efdd-4052-944d-1f7a03a26eca/2/original.jpg",
    "product-p040": "https://m.media-amazon.com/images/I/51umwHo7blL._AC_UF894,1000_QL80_.jpg",
    "product-p042": "https://dm.henkel-dam.com/is/image/henkel/Ceresit-CL-82-ecommerce-UK_global.tif",
    "product-p043": "https://renowa24.pl/hpeciai/c55bc36ece67a559bb3a15f368d59a5a/pol_pl_Styropian-podlogowy-SWISSPOR-EPS-100-036-gr-3-cm-15831_1.webp",
    "product-p044": "https://www.maldrew.com.pl/static/upload/store/Strotex/Mal-drew_-_Szablon_ogolny_(6).png",
    "product-p045": "https://media.knauf.com/a/KaiigP27nUwmNf3ruxp2be",
    "product-p046": "https://www.wienerberger.pl/var/commerce/products/wienerberger/products/wbs_pl_website/wbs_pl_website/wall_acc/856051/assets/asset.imgTransformer/crop_1to1/lg-5/1614355808815/PL_MKT_PSH_POR_Porotherm_25_3x_1_3.jpg",
    "product-p047": "https://mkm-chemiabudowlana.pl/wp-content/uploads/2025/03/weberbase_pas_modelino_D_2017_12_15_wizual.jpg",
    "product-p048": "https://baza.atlas.com.pl/foto/atlas-salta-s_p_1540_20191114_154027.png",
    "product-p050": "https://mera.eu/hpeciai/efc181ed563eed12e9cd127c75ef3421/pol_pl_ZW-330-Atlas-szybkosprawna-zaprawa-wyrownujaca-25kg-175_3.webp",
}

with open('/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/url_checkpoint.json', 'w') as f:
    json.dump(url_map, f, ensure_ascii=False, indent=2)

print(f"Zapisano {len(url_map)} URL-i do checkpoint")


Zapisano 20 URL-i do checkpoint


In [43]:

import json

with open('/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/url_checkpoint.json') as f:
    url_map = json.load(f)

url_map.update({
    "product-p051": "https://eadamex.pl/92056-home_default/tynk-silikonowy-atlas-in-n150-25kg-bialy.jpg",
    "product-p052": "https://dombud.eu/wp-content/uploads/2025/03/MosaicM81.png",
    "product-p053": "https://baza.atlas.com.pl/foto/atlas-cermit-wn_p_2291_20230428_095416.jpg",
    "product-p055": "https://baza.atlas.com.pl/foto/tynk-maszynowy-lekki-cementowo-wapienny-na-zewnatrz_p_806_20250902_124243.png",
    "product-p056": "https://tytan-cdn.tytan.com/uploads/sites/53/2026/03/45171V01_0120do20luster20fix20klej20montaC5BCowy.png",
    "product-p057": "https://pakietbudowlany.pl/474-thickbox_default/ok-klej-uelastyczniony-c1te.jpg",
    "product-p058": "https://budohurt.pl/userdata/public/gfx/23734/CT83-klej-do-styropianu-EPS-STRONG-FIX-25kg.png",
    "product-p059": "https://taniebudowanie24.pl/2679-large_default/ct-190-mw-flex-ceresit-25-kg-klejzaprawa-klejaco-szpachlowa-do-welny-mineralnej.jpg",
    "product-p060": "https://www.profichem24.pl/userdata/public/gfx/12326/035_FUGA_ceramiczna_5kg_lewy_small.png",
    "product-p061": "https://m.media-amazon.com/images/I/517b+6qz1GL.jpg",
})

with open('/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/url_checkpoint.json', 'w') as f:
    json.dump(url_map, f, ensure_ascii=False, indent=2)
print(f"Checkpoint: {len(url_map)} URL-i")


Checkpoint: 30 URL-i
